# Introduction to Pandas

**Pandas** is a Python library for working with structured (tabular) data. It is one of the core tools for data analysis and makes it easy to load, explore, clean, and manipulate data.

This notebook covers the fundamentals:
1. Importing the library
2. The two main data types: `Series` and `DataFrame`
3. Importing and exporting data
4. Describing / summarising data
5. Viewing and selecting data
6. Manipulating data (cleaning, transforming, adding/removing columns)

## 1. Import the Library

By convention, pandas is imported with the alias `pd`. We can check the installed version with `pd.__version__`.

In [ ]:
# Import pandas using the standard "pd" alias
import pandas as pd

# Print the installed pandas version
print(pd.__version__)

## 2. Data Types

Pandas has two core data structures:

- **`Series`** — a one-dimensional column of data (like a single column in a spreadsheet).
- **`DataFrame`** — a two-dimensional table made up of rows and columns (multiple `Series` joined together).

### Series: a 1-dimensional column of data

A `Series` is created from a list of values. Each value is given an automatic index (0, 1, 2, ...).

In [ ]:
# Create a Series of car makes from a list of strings
cars = pd.Series(["BMW", "toyota", "Mercedes"])
cars

In [ ]:
# Create a second Series of colours
colors = pd.Series(["Blue", "red", "pink"])
colors

### DataFrame: rows and columns

A `DataFrame` is built from a dictionary where each key is a column name and each value is the column's data (a list or `Series`). Here we combine the `cars` and `colors` Series into one table.

In [ ]:
# Build a DataFrame from a dictionary of {column name: Series}
car_data = pd.DataFrame({
    "Car type": cars,
    "Color": colors
})
car_data

### Exercises

1. Make a `Series` of different foods.
2. Make a `Series` of different dollar values (these can be integers).
3. Combine your `Series`'s of foods and dollar values into a `DataFrame`.

Try it out for yourself first, then see how your code goes against the solution.

**Note:** Make sure your two `Series` are the same size before combining them in a DataFrame.

#### Solution

In [ ]:
# 1. A Series of different foods
foods = pd.Series(["Pizza", "Sushi", "Salad", "Burger"])

# 2. A Series of dollar values (integers) - same length as foods
prices = pd.Series([12, 18, 9, 14])

# 3. Combine the two Series into a DataFrame
food_data = pd.DataFrame({
    "Food": foods,
    "Price ($)": prices
})
food_data

## 3. Importing Data

Most of the time we don't build DataFrames by hand — we load them from a file. `pd.read_csv()` reads a CSV file into a DataFrame.

In [ ]:
# Read the car sales data from a CSV file into a DataFrame
car_sales = pd.read_csv("../pandas/car-sales.csv")
car_sales

## 4. Exporting Data

We can save a DataFrame back to a CSV file with `.to_csv()`. This is useful after cleaning or transforming the data.

In [ ]:
# Export the DataFrame to a new CSV file
car_sales.to_csv("../pandas/car-sales-exported.csv")

## 5. Describing Data

Pandas provides several methods to quickly understand a dataset:

- `.dtypes` — the data type of each column
- `.describe()` — summary statistics for numeric columns
- `.info()` — column names, non-null counts, and types
- `.mean()`, `.sum()` — aggregate calculations

In [ ]:
# Show the data type of each column
car_sales.dtypes

In [ ]:
# Summary statistics (count, mean, std, min, max, quartiles) for numeric columns
car_sales.describe()

In [ ]:
# Overview of the DataFrame: column names, non-null counts, dtypes, and memory usage
car_sales.info()

In [ ]:
# Calculate the mean of each numeric column (numeric_only avoids errors on text columns)
car_sales.mean(numeric_only=True)

In [ ]:
# The same aggregate methods work directly on a Series
car_prices = pd.Series([10000, 3000, 77000])
car_prices.mean(numeric_only=True)

In [ ]:
# Sum each numeric column
car_sales.sum(numeric_only=True)

## 6. Viewing and Selecting Data

Common ways to view and select parts of a DataFrame:

- `.head()` / `.tail()` — first / last rows
- `.loc[]` — select by **label** (the index value)
- `.iloc[]` — select by **integer position**
- `df["column"]` — select a single column
- Boolean filtering — select rows that meet a condition
- `pd.crosstab()` and `.groupby()` — summarise relationships between columns

In [ ]:
# View the first 5 rows (pass a number to change how many, e.g. head(3))
car_sales.head()

In [ ]:
# View the last 5 rows
car_sales.tail()

### `.loc` vs `.iloc`

A custom index makes the difference clear:

- `.loc[label]` selects by the **index label**.
- `.iloc[position]` selects by the **integer position** (0-based), ignoring the label.

In [ ]:
# Create a Series with a custom (non-sequential) index
animals = pd.Series(["dog", "cat", "horse", "snake", "elephant"],
                    index=[3, 44, 6, 77, 8])
animals

In [ ]:
# .loc uses the index label -> label 44 holds "cat"
animals.loc[44]

In [ ]:
# .iloc uses the position -> position 3 (4th item) holds "snake"
animals.iloc[3]

On the `car_sales` DataFrame the index is the default 0–9, so `.loc[3]` and `.iloc[3]` return the same row. Both return the entire row as a Series.

In [ ]:
# Display the full DataFrame for reference
car_sales

In [ ]:
# Select the row with index label 3
car_sales.loc[3]

In [ ]:
# Select the row at position 3
car_sales.iloc[3]

### Selecting columns

Use `df["column_name"]` to select a single column (returns a `Series`).

In [ ]:
# Select the "Make" column
car_sales["Make"]

In [ ]:
# Select the "Colour" column
car_sales["Colour"]

### Filtering rows with conditions

Passing a boolean condition inside `df[...]` keeps only the rows where the condition is `True`.

In [ ]:
# Keep only rows where the odometer reading is greater than 45,000 KM
car_sales[car_sales["Odometer (KM)"] > 45000]

In [ ]:
# Keep only rows where the Make is exactly "Toyota"
car_sales[car_sales["Make"] == "Toyota"]

### Summarising with crosstab and groupby

- `pd.crosstab()` builds a frequency table counting combinations of two columns.
- `.groupby()` groups rows by a column, then applies an aggregate (here, the mean) to each group.

In [ ]:
# Count how many cars of each Make have each number of Doors
pd.crosstab(car_sales["Make"], car_sales["Doors"])

In [ ]:
# Group rows by Make, then average the numeric columns within each group
car_sales.groupby(["Make"]).mean(numeric_only=True)

### Cleaning the Price column

The `Price` column was loaded as text (`str`) because of the `$` sign and commas (e.g. `"$4,000.00"`). To do maths on it we need to convert it to numbers. The cells below show:

1. Why a naive conversion fails.
2. Inspecting the raw values.
3. The correct fix: strip the `$` and `,` characters, then convert to a numeric type.

In [ ]:
# Note that "Price" is currently stored as a string (str) type
car_sales.info()

In [ ]:
# Naive attempt: converting directly to numeric fails on the "$" and "," characters,
# so every value becomes <NA> (missing). We fix this properly further below.
car_sales['Price'] = pd.to_numeric(car_sales['Price'], errors='coerce').astype('Int64')
car_sales

In [ ]:
# Reload the data and inspect the original Price values to see what's causing the problem
car_sales_check = pd.read_csv("../pandas/car-sales.csv")

print("Original Price column (first 10):")
print(car_sales_check['Price'].head(10))

print("\nPrice dtype:", car_sales_check['Price'].dtype)

# The unique values reveal the "$" and "," characters that block numeric conversion
print("\nUnique values:")
print(car_sales_check['Price'].unique()[:15])

In [ ]:
# Reload the data fresh (the earlier naive conversion turned Price into <NA>)
car_sales = pd.read_csv("../pandas/car-sales.csv")

# Remove the "$" and "," characters, then convert the cleaned text to a numeric type
car_sales['Price'] = car_sales['Price'].str.replace('$', '').str.replace(',', '')
car_sales['Price'] = pd.to_numeric(car_sales['Price']).astype('Int64')

print("Updated Price column:")
print(car_sales[['Make', 'Price']].head(10))

In [ ]:
# Confirm Price is now a numeric (Int64) column
car_sales.info()

In [ ]:
# Preview the cleaned DataFrame
car_sales.head()

## 7. Manipulating Data

This section covers transforming and reshaping data:

- String operations on text columns (e.g. `.str.lower()`)
- Handling missing values (`.fillna()`, `.dropna()`)
- Adding and removing columns
- Sampling rows and resetting the index
- Applying a custom function with `.apply()`

### String operations

The `.str` accessor applies string methods to every value in a text column. Note this returns a **new** Series — the original column is unchanged unless we reassign it.

In [ ]:
# Lowercase every value in the "Make" column (returns a new Series, original unchanged)
car_sales["Make"].str.lower()

In [ ]:
# The "Make" column is still in its original case (the operation above wasn't saved)
car_sales

In [ ]:
# Reassign the column to make the lowercase change permanent
car_sales["Make"] = car_sales["Make"].str.lower()
car_sales.head()

### Handling missing data

Real datasets often have gaps, shown as `NaN` (Not a Number). Two common strategies:

- **Fill** the gaps with a value using `.fillna()` (e.g. the column mean).
- **Drop** rows containing gaps using `.dropna()`.

Here we load a version of the dataset that contains missing values.

In [ ]:
# Load a dataset that contains missing values (shown as NaN)
car_missing_df = pd.read_csv("../pandas/car-sales-missing-data.csv")
car_missing_df

In [ ]:
# Fill missing Odometer values with the column's mean.
# inplace=False returns a new Series without changing the original DataFrame.
car_missing_df["Odometer"].fillna(car_missing_df["Odometer"].mean(), inplace=False)

In [ ]:
# Because we used inplace=False above, the DataFrame still has NaN in Odometer
car_missing_df

In [ ]:
# Anti-pattern demo (raises a ChainedAssignmentError warning - see the note above)
car_missing_df["Odometer"] = car_missing_df["Odometer"].fillna(car_missing_df["Odometer"].mean())

In [ ]:
# Despite the warning, the Odometer NaNs did get filled with the mean in this case
car_missing_df

#### Dropping rows with missing values

`.dropna()` removes any row that still contains a `NaN`. Like other methods, it returns a new DataFrame unless `inplace=True` is used.

In [ ]:
# Drop rows that still contain any NaN (returns a new DataFrame; original unchanged)
car_missing_df.dropna()

In [ ]:
# The original still has all rows (the dropna above was not saved)
car_missing_df

In [ ]:
# inplace=True drops the rows directly in the original DataFrame
car_missing_df.dropna(inplace=True)
car_missing_df

### Adding columns

A new column is created by assigning to `df["new_column"]`. You can assign a `Series`, a list, or a single value (which is repeated for every row).

In [ ]:
# Add a "Seats" column from a Series (lengths must match the number of rows)
seats_column = pd.Series([5, 5, 5, 5, 5, 5])
car_missing_df["Seats"] = seats_column

car_missing_df

In [ ]:
# Add an "Engine size" column from a plain Python list
engine_size = [1.3, 2.4, 3.6, 4.8, 5.2, 2.8]
car_missing_df["ENgine size"] = engine_size

car_missing_df

In [ ]:
# Check the dtypes after adding the new columns
car_missing_df.dtypes

### Creating columns from calculations, and dropping columns

We can build a new column from arithmetic on existing columns, add a constant column, and remove a column with `.drop()` (use `axis=1` for columns).

In [ ]:
# Create a new column by dividing one column by another (price per kilometre)
car_sales["pRICE PER KM"] = car_sales["Price"] / car_sales["Odometer (KM)"]
car_sales

In [ ]:
# Add a constant column: the single value 4 is applied to every row
car_sales["Number of wheels"] = 4
car_sales

In [ ]:
# Remove a column with .drop() (axis=1 means drop a column, not a row)
car_sales = car_sales.drop("pRICE PER KM", axis=1)
car_sales

### Sampling rows and resetting the index

`.sample()` returns a random subset of rows (here 50% via `frac=0.5`), which keeps their original index labels. `.reset_index()` renumbers the rows back to 0, 1, 2, ...

In [ ]:
# Randomly sample 50% of the rows (note the index labels stay shuffled)
car_sales_samples = car_sales.sample(frac=0.5)
car_sales_samples

In [ ]:
# reset_index() returns a new DataFrame with a fresh 0-based index.
# Without inplace=True (or reassignment), car_sales_samples keeps its old index.
car_sales_samples.reset_index()
car_sales_samples

### Applying a custom function with `.apply()`

`.apply()` runs a function on every value in a column. Here we use a `lambda` to convert the odometer reading from kilometres to miles (divide by 1.6).

In [ ]:
# Convert the odometer from kilometres to miles by applying a function to each value
car_sales["Odometer (KM)"] = car_sales["Odometer (KM)"].apply(lambda x: x / 1.6)

In [ ]:
# View the result: odometer values are now in miles
car_sales

## Summary

In this notebook we covered the pandas essentials:

- **Data types** — `Series` (1D) and `DataFrame` (2D).
- **Import / export** — `pd.read_csv()` and `.to_csv()`.
- **Describing data** — `.dtypes`, `.describe()`, `.info()`, `.mean()`, `.sum()`.
- **Viewing / selecting** — `.head()`/`.tail()`, `.loc`/`.iloc`, column selection, boolean filtering, `crosstab`, `groupby`.
- **Manipulating data** — string methods, cleaning text into numbers, handling missing values (`fillna`/`dropna`), adding/dropping columns, sampling, and `.apply()`.

A key takeaway: most pandas methods return a **new** object by default. To change the original DataFrame, either reassign the result (`df["col"] = ...`) or use `inplace=True` where supported.